<h1> Frozen vs Fresh Food - What's Better?? </h1>
<h3> Group Wilbur Atwater | EEP 153 Spring 2026 | Subsistence Diet Cost Project </h3>

In [56]:
# importing libraries
import pandas as pd

# loading necessary external csv files

# dietary minimums for diff sex and age ranges, various micronutrients
mins = pd.read_csv("diet_minimums.csv")
# load dietary maximums for diff sex and age ranges, sodium and calories
maxs = pd.read_csv("diet_maximums.csv")

<h3>[A] Dietary Reference Intakes</h3>
<p>This function takes a person's age and sex as arguments and returns a series of daily reccomended dietary intake amounts for a variety of nutrients. </p>

In [57]:
# use the first column as the nutrient/index column
mins = mins.set_index(mins.columns[0])
maxs = maxs.set_index(maxs.columns[0])

# make sure numbers are numeric
mins = mins.apply(pd.to_numeric, errors="coerce")
maxs = maxs.apply(pd.to_numeric, errors="coerce")

# helper function to determine the right age group
def age_bucket(age):
    age = float(age)
    if age <= 3:
        return "C 1-3"
    elif age <= 8:
        return "4-8"
    elif age <= 13:
        return "9-13"
    elif age <= 18:
        return "14-18"
    elif age <= 30:
        return "19-30"
    elif age <= 50:
        return "31-50"
    else:
        return "51+"

In [67]:
def dietary_requirements(age, sex):
    bucket = age_bucket(age)

    # if the column isnt "C 1-3" then it has to be built using the inputted sex and the age group returned by our helper func
    if bucket == "C 1-3":
        col = "C 1-3"
    else:
        col = f"{sex} {bucket}"

    # minimums + maximums for that column
    s_min = mins[col].copy()
    s_max = maxs[col].copy()

    # combine Energy min + max into one range
    if "Energy" in s_min.index and "Energy" in s_max.index:
        energy_min = s_min["Energy"]
        energy_max = s_max["Energy"]
    
        # convert Series to object so it can hold strings
        s_min = s_min.astype(object)
    
        s_min["Energy"] = f"{int(energy_min)} - {int(energy_max)}"
        s_max = s_max.drop("Energy")

    # label remaining maximums
    s_max = s_max.rename(lambda x: f"{x} (MAX)")

    out = pd.concat([s_min, s_max])
    out.name = f"Requirements for {sex}, age {age} (col={col})"
    return out

In [68]:
# example usages of the code
req = dietary_requirements(age = 40, sex = "F")
req

Nutrition
Energy                            1800 - 3100
Protein                                  46.0
Fiber, total dietary                     25.2
Folate, DFE                             400.0
Calcium, Ca                            1000.0
Carbohydrate, by difference             130.0
Iron, Fe                                 18.0
Magnesium, Mg                           320.0
Niacin                                   14.0
Phosphorus, P                           700.0
Potassium, K                           4700.0
Riboflavin                                1.1
Thiamin                                   1.1
Vitamin A, RAE                          700.0
Vitamin B-12                              2.4
Vitamin B-6                               1.3
Vitamin C, total ascorbic acid           75.0
Vitamin E (alpha-tocopherol)             15.0
Vitamin K (phylloquinone)                90.0
Zinc, Zn                                  8.0
Sodium, Na (MAX)                         2300
Name: Requirements for F

In [70]:
# tests for this function

# test 1: function returns a pandas Series
def test_returns_series():
    result = dietary_requirements(25, "F")
    assert isinstance(result, pd.Series)

# test 2: energy should be displayed as a range
def test_energy_range():
    result = dietary_requirements(25, "F")
    assert "Energy" in result.index
    assert "-" in result["Energy"]

# test 3: ages 1–3 should use the C 1-3 column (no sex prefix)
def test_age_1_3_bucket():
    result = dietary_requirements(2, "F")
    assert "C 1-3" in result.name

# test 4: correct age bucket column is used
def test_age_bucket():
    result = dietary_requirements(30, "M")
    assert "M 19-30" in result.name

# function to run all tests
def run_tests():
    test_returns_series()
    test_energy_range()
    test_age_1_3_bucket()
    test_age_bucket()
    print("All tests passed!")
    
run_tests()

All tests passed!
